# Fabric Notebook Exfiltration Scanner

Scans notebook files (.ipynb, .py, Fabric Item JSON) stored in a Lakehouse Files
directory for potential data exfiltration patterns. Findings are written to a
Delta table in the attached Lakehouse.

## Detection Capabilities
- **AST analysis**: Structural code inspection for dangerous imports, calls, and patterns
- **Regex patterns**: 100+ patterns for URLs, credentials, exfil services, cloud storage
- **Entropy analysis**: Detects obfuscated/encoded payloads
- **Spark-specific**: External writes, streaming sinks, credential access
- **Secret detection**: Hardcoded keys, SAS tokens, passwords, connection strings
- **URL classification**: Read/write direction with 30+ protocol schemes

## Architecture
1. Read files from Lakehouse using `spark.read.format("binaryFile")`
2. Distribute scanning across executors with `mapPartitions`
3. Write findings to a Delta table for downstream analysis

In [ ]:
# === Configuration ===========================================================
# Point this at the Lakehouse Files path containing your notebook exports.
# These may come from Git integration, fabric-cli, or manual export.

# Path under the attached Lakehouse's Files area to scan.
SOURCE_PATH = "Files/notebooks"

# File glob patterns to match. Accepts a single string or a list of patterns.
# Examples:
#   SOURCE_FILE_GLOB = "*.ipynb"              # single pattern
#   SOURCE_FILE_GLOB = ["*.ipynb", "*.py"]    # multiple patterns
#   SOURCE_FILE_GLOB = ["*.ipynb", "*.py", "*.json"]  # three patterns
SOURCE_FILE_GLOB = ["*.ipynb", "*.py"]

# Output table name (written to the default attached Lakehouse).
OUTPUT_TABLE = "notebook_exfil_findings"

# Minimum severity to report: "low", "medium", "high", "critical"
MIN_SEVERITY = "medium"

# Max files to scan (None = all, set an integer for testing)
MAX_FILES = None

# Partition tuning
TARGET_PARTITION_SIZE = 100  # files per Spark partition

## Load Notebook Files from Lakehouse

In [ ]:
from pyspark.sql import functions as F
from functools import reduce

# Normalise SOURCE_FILE_GLOB to a list so single strings work too
_globs = [SOURCE_FILE_GLOB] if isinstance(SOURCE_FILE_GLOB, str) else list(SOURCE_FILE_GLOB)

def _load_glob(pattern):
    return (spark.read.format("binaryFile")
        .option("recursiveFileLookup", "true")
        .option("pathGlobFilter", pattern)
        .load(SOURCE_PATH))

# Load each pattern separately and union; deduplicate by path to avoid double-counting
_dfs = [_load_glob(g) for g in _globs]
file_df = reduce(lambda a, b: a.union(b), _dfs).dropDuplicates(["path"])

n_total = file_df.count()
if n_total == 0:
    raise SystemExit(
        f"No files matched patterns {_globs} under '{SOURCE_PATH}'."
    )

if MAX_FILES:
    file_df = file_df.limit(MAX_FILES)
    n_total = min(n_total, MAX_FILES)

num_partitions = max(1, (n_total + TARGET_PARTITION_SIZE - 1) // TARGET_PARTITION_SIZE)
file_df = file_df.repartition(num_partitions)

print(f"Patterns: {_globs}")
print(f"Found {n_total} files. Scanning in {num_partitions} partition(s).")
print("Sample files:")
for row in file_df.select("path").limit(5).collect():
    print(f"  {row.path}")

## Scanner Engine

All detection logic is inlined so it ships to each Spark executor
without needing external dependencies or file distribution.

In [ ]:
import re
import ast
import json
import math
import base64
from datetime import datetime, timezone
from pyspark.sql import Row

# Severity filtering
SEVERITY_ORDER = ["low", "medium", "high", "critical"]
_MIN_SEV_IDX = SEVERITY_ORDER.index(MIN_SEVERITY.lower())

def _meets_severity(sev):
    return SEVERITY_ORDER.index(sev) >= _MIN_SEV_IDX


# ===== Protocol schemes for URL detection ====================================
URL_PROTOCOL_SCHEMES = [
    "http://", "https://", "ftp://", "ftps://", "sftp://",
    "abfs://", "abfss://", "wasb://", "wasbs://", "adl://",
    "s3://", "s3a://", "s3n://", "gs://",
    "hdfs://", "viewfs://", "dbfs://",
    "jdbc:", "odbc:", "mongodb://", "mongodb+srv://",
    "postgresql://", "postgres://", "mysql://", "mssql://",
    "redis://", "rediss://", "amqp://", "amqps://",
    "kafka://", "kinesis://", "eventhubs://",
    "mqtt://", "mqtts://",
    "ssh://", "scp://", "smb://", "nfs://", "tcp://", "udp://",
]

URL_INDICATORS = URL_PROTOCOL_SCHEMES + [
    ".com", ".net", ".io", ".org", ".dev", ".app",
    ".amazonaws.com", ".blob.core.windows.net", ".dfs.core.windows.net",
    ".database.windows.net", ".azurewebsites.net",
    "/api/", "/v1/", "/v2/", "/v3/", "endpoint", "://",
]

In [ ]:
# ===== Regex detection patterns =============================================
PATTERNS = [
    # --- HTTP exfiltration ---
    (re.compile(r'(requests|httpx|urllib)\s*\.\s*(post|put|patch)\s*\(', re.I),
     "http_exfiltration", "high", "HTTP POST/PUT/PATCH request detected"),
    (re.compile(r'https?://[^\s\'\"]+\.(ngrok|burpcollaborator|requestbin|pipedream|webhook\.site|hookbin|canarytokens|interactsh)', re.I),
     "http_exfiltration", "critical", "Known exfiltration service URL"),
    (re.compile(r'https?://\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}[:/]', re.I),
     "http_exfiltration", "high", "HTTP request to raw IP address"),
    (re.compile(r'urllib\.request\.urlopen\s*\([^,)]+,\s*(data|b["\'])', re.I),
     "http_exfiltration", "high", "urllib.request.urlopen() with data - HTTP POST equivalent"),
    (re.compile(r'urllib\.request\.urlretrieve\s*\(|urlretrieve\s*\(', re.I),
     "http_exfiltration", "medium", "urlretrieve() downloads file to local disk"),

    # --- Cloud storage / file sharing services ---
    (re.compile(r'(api\.dropboxapi\.com|content\.dropboxapi\.com|dropbox\.com/oauth2)', re.I),
     "cloud_storage_write", "high", "Dropbox API endpoint"),
    (re.compile(r'\b(dropbox\.Dropbox|DropboxOAuth2FlowNoRedirect)\s*\(', re.I),
     "cloud_storage_write", "high", "Dropbox SDK client initialization"),
    (re.compile(r'(api\.box\.com|upload\.box\.com|box\.com/api)', re.I),
     "cloud_storage_write", "high", "Box.com API endpoint"),
    (re.compile(r'(drive\.google\.com|www\.googleapis\.com/upload/drive|www\.googleapis\.com/drive)', re.I),
     "cloud_storage_write", "high", "Google Drive API endpoint"),
    (re.compile(r'(mega\.nz|mega\.co\.nz|g\.api\.mega\.co\.nz)', re.I),
     "cloud_storage_write", "high", "MEGA cloud storage endpoint"),
    (re.compile(r'(wetransfer\.com/api|we\.tl)', re.I),
     "cloud_storage_write", "high", "WeTransfer API/URL"),
    (re.compile(r'(api\.pcloud\.com|pcloud\.com/oauth2)', re.I),
     "cloud_storage_write", "high", "pCloud API endpoint"),
    (re.compile(r'(firebaseio\.com|firebasestorage\.googleapis\.com)', re.I),
     "cloud_storage_write", "high", "Firebase endpoint"),
    (re.compile(r'\b\w+\.supabase\.co\b', re.I),
     "cloud_storage_write", "high", "Supabase endpoint"),
    (re.compile(r'(api\.cloudinary\.com|cloudinary\.uploader)', re.I),
     "cloud_storage_write", "high", "Cloudinary upload endpoint"),
    (re.compile(r'(backblazeb2\.com|b2_authorize_account|b2_upload_file)', re.I),
     "cloud_storage_write", "high", "Backblaze B2 storage endpoint"),
    (re.compile(r'(icloud\.com/api|setup\.icloud\.com)', re.I),
     "cloud_storage_write", "high", "iCloud API endpoint"),
    (re.compile(r'\b(upload_blob|upload_file|upload_fileobj|upload_from_string|upload_data|put_object|put_block_blob|put_item|put_record)\s*\(', re.I),
     "cloud_storage_write", "high", "Cloud storage upload SDK call"),

    # --- Paste / gist services ---
    (re.compile(r'(pastebin\.com/api|api\.paste\.ee|dpaste\.com/api|ix\.io|sprunge\.us|hastebin\.com)', re.I),
     "api_exfiltration", "high", "Paste service API - data exfiltration risk"),
    (re.compile(r'api\.github\.com/gists', re.I),
     "api_exfiltration", "high", "GitHub Gists API - data exfiltration risk"),
    (re.compile(r'(gitlab\.com/api.*snippets|snippets\.gitlab\.com)', re.I),
     "api_exfiltration", "high", "GitLab Snippets API"),
    (re.compile(r'api\.airtable\.com', re.I),
     "api_exfiltration", "high", "Airtable API endpoint"),
    (re.compile(r'api\.notion\.(com|so)', re.I),
     "api_exfiltration", "high", "Notion API endpoint"),

    # --- Webhooks ---
    (re.compile(r'https://hooks\.slack\.com/services/', re.I),
     "webhook_exfiltration", "critical", "Slack incoming webhook URL"),
    (re.compile(r'https://discord(app)?\.com/(api/)?webhooks/', re.I),
     "webhook_exfiltration", "critical", "Discord webhook URL"),
    (re.compile(r'https://api\.telegram\.org/bot', re.I),
     "webhook_exfiltration", "critical", "Telegram bot API"),
    (re.compile(r'https://[^\s]*\.execute-api\.[^\s]*\.amazonaws\.com', re.I),
     "webhook_exfiltration", "high", "AWS API Gateway endpoint"),
    (re.compile(r'https://[^\s]+(\.workers\.dev|\.vercel\.app|\.netlify\.app|\.pages\.dev|\.azurewebsites\.net)', re.I),
     "webhook_exfiltration", "high", "Serverless platform endpoint"),

    # --- Microsoft API abuse ---
    (re.compile(r'https://graph\.microsoft\.com/v\d+\.\d+/(me|users|groups|drives|sites)', re.I),
     "api_exfiltration", "high", "Microsoft Graph API call"),
    (re.compile(r'https://management\.azure\.com/subscriptions', re.I),
     "api_exfiltration", "high", "Azure Management API call"),
    (re.compile(r'https://api\.fabric\.microsoft\.com/v1/', re.I),
     "cross_workspace_operation", "high", "Fabric REST API call from within notebook"),
    (re.compile(r'https://api\.powerbi\.com/v1\.0/myorg/datasets/[^/]+/rows', re.I),
     "api_exfiltration", "critical", "Power BI streaming/push dataset API"),
    (re.compile(r'(GraphServiceClient|MSGraph|O365\.Account)\s*\(', re.I),
     "api_exfiltration", "critical", "Microsoft Graph SDK client"),

    # --- Credential access ---
    (re.compile(r'(mssparkutils|notebookutils)\.credentials\.(getSecret|getToken|getConnectionString|getFullConnectionString)', re.I),
     "credential_access", "high", "Fabric credential access"),
    (re.compile(r'(TokenLibrary|dbutils\.secrets)\.(get|getSecret|getToken)', re.I),
     "credential_access", "high", "Token/secret library access"),

    # --- Hardcoded secrets ---
    (re.compile(r'AccountKey\s*=\s*[A-Za-z0-9+/=]{40,}', re.I),
     "credential_access", "critical", "Hardcoded Storage Account Key"),
    (re.compile(r'(\?|&)sig=[A-Za-z0-9%+/=]{20,}', re.I),
     "credential_access", "critical", "SAS token with signature"),
    (re.compile(r'(Password|pwd)\s*=\s*[^;\s\"]{4,}', re.I),
     "credential_access", "high", "Hardcoded SQL password"),
    (re.compile(r'(client[_-]?secret|app[_-]?secret)\s*[=:]\s*["\']?[A-Za-z0-9~._-]{16,}', re.I),
     "credential_access", "critical", "Hardcoded client/app secret"),
    (re.compile(r'["\']Bearer\s+eyJ[A-Za-z0-9_-]{20,}', re.I),
     "credential_access", "critical", "Hardcoded Bearer JWT token"),
    (re.compile(r'SharedAccessKey\s*=\s*[A-Za-z0-9+/=]{30,}', re.I),
     "credential_access", "critical", "Hardcoded Event Hub / Service Bus key"),
    (re.compile(r'(openai[_-]?api[_-]?key|OPENAI_API_KEY)\s*[=:]\s*["\']?sk-[A-Za-z0-9]{20,}', re.I),
     "credential_access", "critical", "Hardcoded OpenAI API key"),
    (re.compile(r'(connection[_-]?string|conn[_-]?str)\s*[=:]\s*["\'][^\"{20,}["\']', re.I),
     "credential_access", "high", "Hardcoded connection string"),

    # --- Shell / package install ---
    (re.compile(r'^!pip\s+install\b|^%pip\s+install\b|^%conda\s+install\b', re.I | re.MULTILINE),
     "in_notebook_package_install", "high", "In-notebook package installation"),
    (re.compile(r'pip\s+install\s+.*--(index-url|extra-index-url)\s+http', re.I),
     "in_notebook_package_install", "critical", "pip install from non-PyPI index"),
    (re.compile(r'^!(curl|wget|nc|ncat|netcat|scp|rsync|ssh|sftp|ftp)\b', re.I | re.MULTILINE),
     "shell_execution", "critical", "Shell command for network file transfer"),
    (re.compile(r'(os\.system|os\.popen|subprocess\.(run|Popen|call|check_output))\s*\(', re.I),
     "shell_execution", "high", "Shell command execution"),

    # --- Spark external writes ---
    (re.compile(r'\.format\s*\(\s*["\'](?:com\.mongodb|org\.apache\.spark\.sql\.cassandra|kafka|es\.spark)', re.I),
     "spark_external_write", "high", "Spark write with external connector format"),
    (re.compile(r'writeStream.*\.format\s*\(\s*["\'](?:kafka|eventhubs|socket|mqtt|kinesis)["\']\s*\)', re.I),
     "spark_streaming_exfiltration", "critical", "Spark streaming to external broker"),
    (re.compile(r'writeStream.*foreachBatch', re.I),
     "spark_streaming_exfiltration", "high", "Spark streaming foreachBatch - custom sink"),
    (re.compile(r'spark\.conf\.set\s*\(\s*["\'].*(account\.key|access\.key|secret|password|token)', re.I),
     "credential_access", "high", "Spark config with external credentials"),
    (re.compile(r'(dbutils\.fs\.cp|mssparkutils\.fs\.cp|shutil\.copy).*(?:s3|wasb|gs|abfss|adl|ftp)', re.I),
     "filesystem_exfiltration", "critical", "File copy to external cloud storage"),
    (re.compile(r'(dbutils\.fs\.mount|mssparkutils\.fs\.mount)', re.I),
     "spark_external_write", "high", "Mounting external storage"),
    (re.compile(r'(mssparkutils|notebookutils)\.fs\.(put|append|mv)\s*\(', re.I),
     "filesystem_exfiltration", "high", "mssparkutils/notebookutils file write/move"),

    # --- JDBC / database ---
    (re.compile(r'jdbc:(mysql|postgresql|sqlserver|oracle|db2|redshift|snowflake|bigquery)://[^\s\'\"]+', re.I),
     "external_database_write", "high", "JDBC connection string to external database"),
    (re.compile(r'Server\s*=\s*[^;]+\.database\.windows\.net', re.I),
     "external_database_write", "high", "Azure SQL connection string"),

    # --- Email ---
    (re.compile(r'(smtplib\.SMTP|MIMEMultipart|MIMEText|send_message|sendmail)\s*[\(\.]', re.I),
     "email_exfiltration", "high", "Email sending capability"),
    (re.compile(r'(api\.sendgrid\.com|api\.mailgun\.net|mandrillapp\.com/api|api\.sparkpost\.com)', re.I),
     "email_exfiltration", "high", "Transactional email API endpoint"),

    # --- DNS exfiltration ---
    (re.compile(r'(dns\.resolver|dnspython|socket\.getaddrinfo)', re.I),
     "dns_exfiltration", "high", "DNS resolution - potential DNS exfiltration"),

    # --- Encoding / obfuscation ---
    (re.compile(r'(exec|eval)\s*\(\s*(base64|codecs|bytes\.fromhex|bytearray)', re.I),
     "dynamic_code_execution", "critical", "Execution of decoded/deobfuscated code"),
    (re.compile(r'getattr\s*\(\s*__builtins__', re.I),
     "dynamic_code_execution", "critical", "Dynamic access to builtins - obfuscation"),

    # --- Polars / Pandas writes ---
    (re.compile(r'\.write_(?:delta|parquet|csv|json|ndjson|ipc|arrow|avro|excel|database)\s*\(', re.I),
     "filesystem_exfiltration", "medium", "Polars DataFrame write to file/database"),
    (re.compile(r'\.to_(?:csv|json|parquet|sql|excel|html|hdf|pickle|orc|feather|stata|gbq|xml)\s*\(', re.I),
     "filesystem_exfiltration", "medium", "Pandas DataFrame export to file/database"),

    # --- SQL DML ---
    (re.compile(r'\.execute\s*\(\s*[f"\']*(INSERT\s+INTO|UPDATE\s+|DELETE\s+FROM|MERGE\s+INTO)', re.I),
     "external_database_write", "medium", "SQL DML via .execute()"),
    (re.compile(r'spark\.sql\s*\(\s*f?["\']*(INSERT\s+INTO|UPDATE\s+|DELETE\s+FROM|MERGE\s+INTO)', re.I),
     "spark_external_write", "medium", "spark.sql() with data-modifying statement"),
]

In [ ]:
# ===== Suspicious imports ===================================================
SUSPICIOUS_IMPORTS = {
    "requests", "httpx", "urllib", "urllib3", "http.client", "aiohttp", "pycurl",
    "boto3", "botocore", "azure.storage.blob", "azure.storage.filedatalake",
    "google.cloud.storage", "gcsfs", "s3fs", "adlfs",
    "paramiko", "ftplib", "pysftp", "smtplib",
    "socket", "websocket", "websockets",
    "grpc", "grpcio",
    "pymongo", "kafka", "confluent_kafka", "pika",
    "azure.eventhub", "azure.servicebus",
    "telegram", "slack_sdk", "discord", "twilio", "sendgrid",
    "msgraph", "O365", "office365",
    "ctypes", "cffi",
    "dropbox", "boxsdk", "pydrive", "pydrive2", "googleapiclient",
    "mega", "firebase_admin", "pyrebase", "supabase", "cloudinary", "b2sdk",
    "pyperclip", "clipboard",
}

IMPORT_CATEGORIES = {
    "requests": "http_exfiltration", "httpx": "http_exfiltration",
    "urllib": "http_exfiltration", "aiohttp": "http_exfiltration",
    "boto3": "cloud_storage_write", "botocore": "cloud_storage_write",
    "azure.storage.blob": "cloud_storage_write",
    "google.cloud.storage": "cloud_storage_write",
    "paramiko": "filesystem_exfiltration", "ftplib": "filesystem_exfiltration",
    "smtplib": "email_exfiltration",
    "socket": "raw_socket_network", "websocket": "raw_socket_network",
    "pymongo": "external_database_write", "kafka": "external_database_write",
    "telegram": "webhook_exfiltration", "slack_sdk": "webhook_exfiltration",
    "discord": "webhook_exfiltration",
    "msgraph": "api_exfiltration", "O365": "api_exfiltration",
    "dropbox": "cloud_storage_write", "boxsdk": "cloud_storage_write",
    "pydrive2": "cloud_storage_write", "googleapiclient": "cloud_storage_write",
    "mega": "cloud_storage_write", "firebase_admin": "cloud_storage_write",
    "supabase": "cloud_storage_write", "cloudinary": "cloud_storage_write",
    "ctypes": "native_code_execution", "cffi": "native_code_execution",
}


# ===== URL detection with direction classification ==========================
_scheme_prefixes = set()
for _s in URL_PROTOCOL_SCHEMES:
    _p = _s.rstrip(":/")
    if _p:
        _scheme_prefixes.add(re.escape(_p))
_schemes_alt = "|".join(sorted(_scheme_prefixes, key=len, reverse=True))

URL_RE = re.compile(
    rf'(?:["\'\'`])?(?P<url>(?:{_schemes_alt})s?://[^\s"\'\'`<>)\]}},;]+|'
    rf'(?:{_schemes_alt})s?:[^\s"\'\'`<>)\]}},;]+|'
    rf'\bwww\.[^\s"\'\'`<>)\]}},;]+\.[^\s"\'\'`<>)\]}},;]+)',
    re.I
)
TRAILING_CHARS = '.,;:!?)]}>\"\'`'

BENIGN_URL_PATTERNS = [
    re.compile(r'https?://(localhost|127\.0\.0\.1|0\.0\.0\.0)(:|/|$)', re.I),
    re.compile(r'https?://json\.schemastore\.org/', re.I),
    re.compile(r'https?://schemas?\.(microsoft|xmlsoap|w3)\.', re.I),
    re.compile(r'https?://(pypi|pip|conda|anaconda)\.(org|io)/', re.I),
    re.compile(r'https?://(docs|learn|aka)\.(microsoft|python|spark)\.', re.I),
    re.compile(r'https?://spark\.apache\.org/', re.I),
    re.compile(r'^file://', re.I),
]

READ_CONTEXT_RE = re.compile(
    r'(?:\brequests?\.(?:get|head|options)\s*\(|\bspark\.read\b|\.read_\w+\s*\(|\bdownload\w*\s*\(|!\s*wget\b)',
    re.I
)
WRITE_CONTEXT_RE = re.compile(
    r'(?:\brequests?\.(?:post|put|patch|delete)\s*\(|\.write\.\w+|\.save\s*\(|\.to_\w+\s*\(|\bupload\w*\s*\()',
    re.I
)


# ===== Entropy analysis =====================================================
def _shannon_entropy(s):
    if not s:
        return 0.0
    freq = {}
    for c in s:
        freq[c] = freq.get(c, 0) + 1
    n = len(s)
    return -sum((count / n) * math.log2(count / n) for count in freq.values())

In [ ]:
# ===== Code extraction =====================================================
def _extract_code(content_bytes, file_path):
    """Extract code blocks from a file. Returns list of (code_str, cell_index)."""
    try:
        text = content_bytes.decode("utf-8", errors="ignore")
    except Exception:
        return []

    if file_path.endswith(".py"):
        return [(text, 0)]

    if file_path.endswith(".ipynb") or file_path.endswith(".json"):
        try:
            data = json.loads(text)
        except (json.JSONDecodeError, ValueError):
            return []

        if not isinstance(data, dict):
            return []

        # Standard .ipynb format
        if "cells" in data:
            blocks = []
            for i, cell in enumerate(data.get("cells", [])):
                if cell.get("cell_type") == "code":
                    src = cell.get("source", [])
                    code_str = "".join(src) if isinstance(src, list) else str(src)
                    blocks.append((code_str, i))
            return blocks

        # Fabric Item JSON with definition.parts
        parts = (data.get("definition") or {}).get("parts") or []
        if parts:
            blocks = []
            for i, part in enumerate(parts):
                path = part.get("path", "")
                if path.endswith(".py") or path.endswith(".ipynb"):
                    try:
                        payload = base64.b64decode(
                            part.get("payload", "")
                        ).decode("utf-8", errors="ignore")
                        if path.endswith(".ipynb"):
                            nb = json.loads(payload)
                            for j, cell in enumerate(nb.get("cells", [])):
                                if cell.get("cell_type") == "code":
                                    src = cell.get("source", [])
                                    blocks.append((
                                        "".join(src) if isinstance(src, list) else str(src), j
                                    ))
                        else:
                            blocks.append((payload, i))
                    except Exception:
                        continue
            return blocks

        # Fabric Item JSON with properties.cells
        cells = (data.get("properties") or {}).get("cells") or []
        if cells:
            blocks = []
            for i, cell in enumerate(cells):
                if cell.get("cell_type") == "code":
                    src = cell.get("source", [])
                    blocks.append(("".join(src) if isinstance(src, list) else str(src), i))
            return blocks

    return []

In [ ]:
# ===== AST-based detection ==================================================
def _ast_scan(code_str):
    """Run lightweight AST checks. Returns list of finding dicts."""
    findings = []
    try:
        tree = ast.parse(code_str)
    except SyntaxError:
        return findings

    for node in ast.walk(tree):
        # Check imports
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            if isinstance(node, ast.Import):
                mods = [a.name for a in node.names]
            else:
                mods = [node.module] if node.module else []
            for mod in mods:
                if not mod:
                    continue
                top = mod.split(".")[0]
                matched = mod if mod in SUSPICIOUS_IMPORTS else (
                    top if top in SUSPICIOUS_IMPORTS else None)
                if matched:
                    cat = IMPORT_CATEGORIES.get(matched, "suspicious_import")
                    findings.append({
                        "category": cat, "severity": "high",
                        "message": f"Import of suspicious module: {mod}",
                        "line": getattr(node, "lineno", 0),
                    })

        # Check dangerous calls: eval/exec/compile
        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name) and node.func.id in ("eval", "exec", "compile"):
                findings.append({
                    "category": "dynamic_code_execution", "severity": "critical",
                    "message": f"{node.func.id}() call - dynamic code execution",
                    "line": node.lineno,
                })

    return findings


# ===== Full scan of one code block ==========================================
def _scan_code(code_str, cell_idx):
    """Apply all detection engines to a single code block."""
    findings = []
    lines = code_str.splitlines()

    # 1. Regex patterns
    for pattern, category, severity, message in PATTERNS:
        if not _meets_severity(severity):
            continue
        for m in pattern.finditer(code_str):
            line_no = code_str[:m.start()].count("\n") + 1
            snippet = lines[line_no - 1].strip()[:120] if line_no <= len(lines) else ""
            findings.append({
                "category": category, "severity": severity,
                "message": message, "line": line_no,
                "code_snippet": snippet,
            })

    # 2. AST scan
    findings.extend(_ast_scan(code_str))

    # 3. URL literal detection with direction
    if _meets_severity("medium"):
        seen_urls = set()
        for m in URL_RE.finditer(code_str):
            url = m.group("url")
            while url and url[-1] in TRAILING_CHARS:
                url = url[:-1]
            if not url:
                continue
            url_lower = url.lower().rstrip("/")
            if url_lower in seen_urls:
                continue
            if any(bp.search(url) for bp in BENIGN_URL_PATTERNS):
                continue
            seen_urls.add(url_lower)
            # Direction classification
            ctx_start = max(0, m.start() - 300)
            ctx = code_str[ctx_start:m.start()]
            has_read = READ_CONTEXT_RE.search(ctx)
            has_write = WRITE_CONTEXT_RE.search(ctx)
            if has_read and has_write:
                direction = "read_write"
            elif has_write:
                direction = "write"
            elif has_read:
                direction = "read"
            else:
                direction = "unknown"
            line_no = code_str[:m.start()].count("\n") + 1
            findings.append({
                "category": "external_url_reference", "severity": "medium",
                "message": f"External URL/URI [direction: {direction}]: {url[:100]}",
                "line": line_no,
                "code_snippet": lines[line_no - 1].strip()[:120] if line_no <= len(lines) else "",
            })

    # 4. Entropy analysis
    if _meets_severity("high"):
        try:
            tree = ast.parse(code_str)
            for node in ast.walk(tree):
                if isinstance(node, ast.Constant) and isinstance(node.value, str):
                    s = node.value
                    if len(s) > 40:
                        ent = _shannon_entropy(s)
                        if ent > 4.5 and not s.startswith(("SELECT", "INSERT", "CREATE", "ALTER")):
                            findings.append({
                                "category": "encoding_obfuscation", "severity": "high",
                                "message": f"High-entropy string (entropy={ent:.2f})",
                                "line": getattr(node, "lineno", 0),
                                "code_snippet": s[:60] + "...",
                            })
        except SyntaxError:
            pass

    return findings

In [ ]:
# ===== Partition processing function ========================================
def scan_partition(rows):
    """Process a batch of binary file rows. Yields Row objects for findings."""
    now = datetime.now(timezone.utc)
    out = []

    for row in rows:
        file_path = row.path
        content_bytes = bytes(row.content)
        fname = file_path.rsplit("/", 1)[-1] if "/" in file_path else file_path

        code_blocks = _extract_code(content_bytes, fname)
        if not code_blocks:
            continue

        seen = set()
        for code_str, cell_idx in code_blocks:
            if not code_str or not code_str.strip():
                continue
            findings = _scan_code(code_str, cell_idx)

            for f in findings:
                key = (fname, cell_idx, f.get("line"), f["category"], f["message"])
                if key in seen:
                    continue
                seen.add(key)

                if not _meets_severity(f["severity"]):
                    continue

                out.append(Row(
                    file_path=file_path,
                    file_name=fname,
                    cell_index=int(cell_idx),
                    line_number=int(f.get("line", 0)),
                    category=f["category"],
                    severity=f["severity"],
                    message=f["message"],
                    code_snippet=f.get("code_snippet", "")[:200],
                    scanned_at=now,
                ))

    return iter(out)

## Execute Scan

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, TimestampType
)

scan_schema = StructType([
    StructField("file_path", StringType(), True),
    StructField("file_name", StringType(), True),
    StructField("cell_index", IntegerType(), True),
    StructField("line_number", IntegerType(), True),
    StructField("category", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("message", StringType(), True),
    StructField("code_snippet", StringType(), True),
    StructField("scanned_at", TimestampType(), True),
])

findings_df = file_df.rdd.mapPartitions(scan_partition).toDF(schema=scan_schema)
findings_df.cache()

n_findings = findings_df.count()
print(f"Scan complete. {n_findings} findings detected across {n_total} files.")

## Write Results to Lakehouse Table

In [ ]:
findings_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(OUTPUT_TABLE)

print(f"Results written to table: {OUTPUT_TABLE}")
print(f"Total findings: {n_findings}")

## Summary View

In [ ]:
from pyspark.sql import functions as F

print("=" * 60)
print("FINDINGS SUMMARY")
print("=" * 60)

# By severity
print("\nBy Severity:")
(findings_df
    .groupBy("severity")
    .agg(F.count("*").alias("count"))
    .orderBy(F.when(F.col("severity") == "critical", 0)
              .when(F.col("severity") == "high", 1)
              .when(F.col("severity") == "medium", 2)
              .otherwise(3))
    .show(truncate=False))

# By category
print("By Category:")
(findings_df
    .groupBy("category")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
    .show(50, truncate=False))

# Top findings
print("\nTop Findings (CRITICAL + HIGH):")
(findings_df
    .filter(F.col("severity").isin("critical", "high"))
    .select("severity", "category", "file_name", "line_number", "message")
    .orderBy(F.when(F.col("severity") == "critical", 0).otherwise(1), "file_name")
    .show(50, truncate=False))

## Query Results (Optional)

Use this cell to query the findings table interactively after the scan completes.

In [ ]:
# Example queries against the findings table
# Uncomment the query you want to run:

# All critical findings:
# spark.sql(f"SELECT * FROM {OUTPUT_TABLE} WHERE severity = 'critical'").show(truncate=False)

# Findings by file:
# spark.sql(f"SELECT file_name, COUNT(*) as finding_count FROM {OUTPUT_TABLE} GROUP BY file_name ORDER BY finding_count DESC").show(truncate=False)

# Credential findings:
# spark.sql(f"SELECT * FROM {OUTPUT_TABLE} WHERE category = 'credential_access'").show(truncate=False)

# External URLs detected:
# spark.sql(f"SELECT * FROM {OUTPUT_TABLE} WHERE category = 'external_url_reference'").show(truncate=False)

# Cloud storage exfil attempts:
# spark.sql(f"SELECT * FROM {OUTPUT_TABLE} WHERE category = 'cloud_storage_write'").show(truncate=False)